# Base05 — criação do `ID_GLEBA2` por geometria

Este notebook lê a base detalhada produzida pelo Base04 e cria a coluna `ID_GLEBA2` sem excluir ou agrupar linhas.

Regra aplicada:

- a primeira ocorrência de uma geometria conserva o seu `ID_GLEBA`;
- toda repetição **exatamente igual** da geometria recebe o mesmo `ID_GLEBA2` da primeira ocorrência;
- uma geometria nova conserva o seu próprio `ID_GLEBA`;
- o arquivo original não é alterado.

Exemplo: se os IDs 22.554 e 126.293 tiverem o mesmo WKT, os dois receberão `ID_GLEBA2 = 22.554`. Como IDs de geometrias repetidas são reutilizados, a sequência final poderá conter lacunas.

In [1]:
# ============================================================
# 1. MONTAR O GOOGLE DRIVE E IMPORTAR BIBLIOTECAS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import csv
import gc
import hashlib
import os
import pandas as pd

pd.set_option('display.max_columns', 200)
print('Bibliotecas carregadas com sucesso.')

Mounted at /content/drive
Bibliotecas carregadas com sucesso.


In [2]:
# ============================================================
# 2. CAMINHOS E CONFIGURAÇÕES
# ============================================================

ARQUIVO_ENTRADA = Path(
    '/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/'
    'Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/'
    'sicor_cerrado_2021_2025_enriquecido_detalhado.csv'
)

ARQUIVO_SAIDA = ARQUIVO_ENTRADA.with_name(
    'sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv'
)

ARQUIVO_TEMPORARIO = ARQUIVO_ENTRADA.with_name(
    'TEMP_sicor_cerrado_2021_2025_com_id_gleba2.csv'
)

COLUNA_ID_ORIGINAL = 'ID_GLEBA'
COLUNA_ID_NOVA = 'ID_GLEBA2'
COLUNA_GEOMETRIA = 'GT_GEOMETRIA'
TAMANHO_BLOCO = 20_000

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f'Arquivo de entrada não encontrado: {ARQUIVO_ENTRADA}')

ARQUIVO_SAIDA.parent.mkdir(parents=True, exist_ok=True)

print('Entrada:', ARQUIVO_ENTRADA)
print('Saída:  ', ARQUIVO_SAIDA)
print('O arquivo original será preservado.')

Entrada: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado.csv
Saída:   /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv
O arquivo original será preservado.


In [3]:
# ============================================================
# 3. IDENTIFICAR O SEPARADOR E VALIDAR O CABEÇALHO
# ============================================================

def identificar_separador(caminho):
    # Usa somente o cabeçalho. Assim, as muitas vírgulas do WKT não
    # interferem na identificação do separador do CSV.
    with open(caminho, 'r', encoding='utf-8-sig', newline='') as arquivo:
        primeira_linha = arquivo.readline()

    candidatos = [';', ',', '\t', '|']
    separador = max(candidatos, key=primeira_linha.count)

    if primeira_linha.count(separador) == 0:
        raise ValueError('Não foi possível identificar o separador do CSV.')

    return separador

SEPARADOR = identificar_separador(ARQUIVO_ENTRADA)

cabecalho = pd.read_csv(
    ARQUIVO_ENTRADA,
    sep=SEPARADOR,
    encoding='utf-8-sig',
    nrows=0
)

colunas_obrigatorias = {
    COLUNA_ID_ORIGINAL,
    COLUNA_GEOMETRIA
}
colunas_ausentes = colunas_obrigatorias - set(cabecalho.columns)

if colunas_ausentes:
    raise KeyError(
        'Colunas obrigatórias ausentes: ' + ', '.join(sorted(colunas_ausentes))
    )

print('Separador identificado:', repr(SEPARADOR))
print('Quantidade de colunas na entrada:', len(cabecalho.columns))
print('Colunas necessárias encontradas.')

Separador identificado: ';'
Quantidade de colunas na entrada: 26
Colunas necessárias encontradas.


## Primeira passagem: identificar o ID original de cada geometria

Para economizar memória, esta etapa lê somente `ID_GLEBA` e `GT_GEOMETRIA`, em blocos. O texto WKT é convertido em uma assinatura SHA-256; geometrias com WKT exatamente igual produzem a mesma assinatura. O menor `ID_GLEBA` encontrado para cada assinatura será o `ID_GLEBA2`.

In [4]:
# ============================================================
# 4. MAPEAR CADA GEOMETRIA AO MENOR ID_GLEBA
# ============================================================

def assinatura_geometria(wkt):
    return hashlib.sha256(wkt.encode('utf-8')).digest()

id_canonico_por_geometria = {}
total_registros = 0
total_sem_geometria = 0
ids_originais_encontrados = set()

leitor_mapeamento = pd.read_csv(
    ARQUIVO_ENTRADA,
    sep=SEPARADOR,
    encoding='utf-8-sig',
    usecols=[COLUNA_ID_ORIGINAL, COLUNA_GEOMETRIA],
    dtype='string',
    keep_default_na=False,
    chunksize=TAMANHO_BLOCO,
    low_memory=False
)

for numero_bloco, bloco in enumerate(leitor_mapeamento, start=1):
    ids_numericos = pd.to_numeric(
        bloco[COLUNA_ID_ORIGINAL],
        errors='coerce'
    )

    if ids_numericos.isna().any():
        linhas_invalidas = bloco.loc[
            ids_numericos.isna(),
            COLUNA_ID_ORIGINAL
        ].head(10).tolist()
        raise ValueError(
            f'Foram encontrados ID_GLEBA inválidos. Exemplos: {linhas_invalidas}'
        )

    ids_numericos = ids_numericos.astype('int64')

    if ids_numericos.duplicated().any():
        raise RuntimeError('Foram encontrados ID_GLEBA duplicados no mesmo bloco.')

    ids_do_bloco = set(ids_numericos.tolist())
    if ids_originais_encontrados.intersection(ids_do_bloco):
        raise RuntimeError('Foram encontrados ID_GLEBA duplicados na base de entrada.')
    ids_originais_encontrados.update(ids_do_bloco)

    geometrias = bloco[COLUNA_GEOMETRIA].astype('string')
    mascara_valida = geometrias.str.strip().ne('')
    total_sem_geometria += int((~mascara_valida).sum())

    for id_original, geometria in zip(
        ids_numericos[mascara_valida],
        geometrias[mascara_valida]
    ):
        assinatura = assinatura_geometria(geometria)
        id_atual = id_canonico_por_geometria.get(assinatura)

        if id_atual is None or id_original < id_atual:
            id_canonico_por_geometria[assinatura] = int(id_original)

    total_registros += len(bloco)

    print(
        f'Bloco {numero_bloco:03d}: '        f'{total_registros:,} registros lidos | '        f'{len(id_canonico_por_geometria):,} geometrias distintas'
    )

    del bloco, ids_numericos, geometrias, mascara_valida
    gc.collect()

total_geometrias_validas = total_registros - total_sem_geometria
total_geometrias_distintas = len(id_canonico_por_geometria)
total_ocorrencias_repetidas = (
    total_geometrias_validas - total_geometrias_distintas
)

print('\nMapeamento concluído.')
print('Registros:', f'{total_registros:,}')
print('Geometrias distintas:', f'{total_geometrias_distintas:,}')
print('Ocorrências adicionais de geometrias repetidas:', f'{total_ocorrencias_repetidas:,}')
print('Registros sem geometria:', f'{total_sem_geometria:,}')

Bloco 001: 20,000 registros lidos | 18,663 geometrias distintas
Bloco 002: 40,000 registros lidos | 36,353 geometrias distintas
Bloco 003: 60,000 registros lidos | 53,609 geometrias distintas
Bloco 004: 80,000 registros lidos | 70,670 geometrias distintas
Bloco 005: 100,000 registros lidos | 87,376 geometrias distintas
Bloco 006: 120,000 registros lidos | 101,740 geometrias distintas
Bloco 007: 140,000 registros lidos | 116,875 geometrias distintas
Bloco 008: 160,000 registros lidos | 132,105 geometrias distintas
Bloco 009: 180,000 registros lidos | 147,059 geometrias distintas
Bloco 010: 200,000 registros lidos | 163,042 geometrias distintas
Bloco 011: 220,000 registros lidos | 179,509 geometrias distintas
Bloco 012: 240,000 registros lidos | 195,275 geometrias distintas
Bloco 013: 260,000 registros lidos | 211,292 geometrias distintas
Bloco 014: 280,000 registros lidos | 226,847 geometrias distintas
Bloco 015: 300,000 registros lidos | 243,097 geometrias distintas
Bloco 016: 320,000 

## Segunda passagem: criar o novo CSV

A base completa é relida em blocos. A coluna `ID_GLEBA2` é inserida imediatamente após `ID_GLEBA`. Nenhuma linha é removida. Registros eventualmente sem geometria conservam o próprio `ID_GLEBA` como `ID_GLEBA2`.

In [5]:
# ============================================================
# 5. GRAVAR A BASE COMPLETA COM ID_GLEBA2
# ============================================================

if ARQUIVO_TEMPORARIO.exists():
    ARQUIVO_TEMPORARIO.unlink()

total_gravado = 0
escrever_cabecalho = True

leitor_saida = pd.read_csv(
    ARQUIVO_ENTRADA,
    sep=SEPARADOR,
    encoding='utf-8-sig',
    dtype='string',
    keep_default_na=False,
    chunksize=TAMANHO_BLOCO,
    low_memory=False
)

for numero_bloco, bloco in enumerate(leitor_saida, start=1):
    if COLUNA_ID_NOVA in bloco.columns:
        bloco = bloco.drop(columns=[COLUNA_ID_NOVA])

    ids_originais = pd.to_numeric(
        bloco[COLUNA_ID_ORIGINAL],
        errors='raise'
    ).astype('int64')

    geometrias = bloco[COLUNA_GEOMETRIA].astype('string')
    mascara_valida = geometrias.str.strip().ne('')

    novos_ids = ids_originais.copy()
    novos_ids.loc[mascara_valida] = [
        id_canonico_por_geometria[assinatura_geometria(geometria)]
        for geometria in geometrias[mascara_valida]
    ]

    posicao_id_original = bloco.columns.get_loc(COLUNA_ID_ORIGINAL)
    bloco.insert(
        posicao_id_original + 1,
        COLUNA_ID_NOVA,
        novos_ids.astype('int64')
    )

    bloco.to_csv(
        ARQUIVO_TEMPORARIO,
        mode='a',
        sep=SEPARADOR,
        encoding='utf-8-sig' if escrever_cabecalho else 'utf-8',
        index=False,
        header=escrever_cabecalho,
        quoting=csv.QUOTE_MINIMAL
    )

    total_gravado += len(bloco)
    escrever_cabecalho = False

    print(
        f'Bloco {numero_bloco:03d}: '        f'{total_gravado:,} registros gravados'
    )

    del bloco, ids_originais, geometrias, mascara_valida, novos_ids
    gc.collect()

if total_gravado != total_registros:
    raise RuntimeError(
        f'Entrada com {total_registros:,} registros, mas foram gravados '        f'{total_gravado:,}.'
    )

os.replace(ARQUIVO_TEMPORARIO, ARQUIVO_SAIDA)

print('\nArquivo criado com sucesso:')
print(ARQUIVO_SAIDA)

Bloco 001: 20,000 registros gravados
Bloco 002: 40,000 registros gravados
Bloco 003: 60,000 registros gravados
Bloco 004: 80,000 registros gravados
Bloco 005: 100,000 registros gravados
Bloco 006: 120,000 registros gravados
Bloco 007: 140,000 registros gravados
Bloco 008: 160,000 registros gravados
Bloco 009: 180,000 registros gravados
Bloco 010: 200,000 registros gravados
Bloco 011: 220,000 registros gravados
Bloco 012: 240,000 registros gravados
Bloco 013: 260,000 registros gravados
Bloco 014: 280,000 registros gravados
Bloco 015: 300,000 registros gravados
Bloco 016: 320,000 registros gravados
Bloco 017: 340,000 registros gravados
Bloco 018: 360,000 registros gravados
Bloco 019: 380,000 registros gravados
Bloco 020: 400,000 registros gravados
Bloco 021: 420,000 registros gravados
Bloco 022: 440,000 registros gravados
Bloco 023: 460,000 registros gravados
Bloco 024: 480,000 registros gravados
Bloco 025: 480,679 registros gravados

Arquivo criado com sucesso:
/content/drive/MyDrive/Co

In [6]:
# ============================================================
# 6. VALIDAR O ARQUIVO FINAL
# ============================================================

total_validado = 0
ids_gleba2_distintos = set()
divergencias = 0

leitor_validacao = pd.read_csv(
    ARQUIVO_SAIDA,
    sep=SEPARADOR,
    encoding='utf-8-sig',
    usecols=[COLUNA_ID_ORIGINAL, COLUNA_ID_NOVA, COLUNA_GEOMETRIA],
    dtype='string',
    keep_default_na=False,
    chunksize=TAMANHO_BLOCO,
    low_memory=False
)

for bloco in leitor_validacao:
    ids_originais = pd.to_numeric(
        bloco[COLUNA_ID_ORIGINAL], errors='raise'
    ).astype('int64')
    ids_novos = pd.to_numeric(
        bloco[COLUNA_ID_NOVA], errors='raise'
    ).astype('int64')
    geometrias = bloco[COLUNA_GEOMETRIA].astype('string')
    mascara_valida = geometrias.str.strip().ne('')

    esperados = ids_originais.copy()
    esperados.loc[mascara_valida] = [
        id_canonico_por_geometria[assinatura_geometria(geometria)]
        for geometria in geometrias[mascara_valida]
    ]

    divergencias += int((ids_novos != esperados).sum())
    ids_gleba2_distintos.update(ids_novos.tolist())
    total_validado += len(bloco)

    del bloco, ids_originais, ids_novos, geometrias, mascara_valida, esperados
    gc.collect()

if total_validado != total_registros:
    raise RuntimeError('A quantidade de registros do arquivo final está incorreta.')

if divergencias != 0:
    raise RuntimeError(
        f'Foram encontradas {divergencias:,} divergências em ID_GLEBA2.'
    )

total_ids_gleba2_esperado = total_geometrias_distintas + total_sem_geometria
if len(ids_gleba2_distintos) != total_ids_gleba2_esperado:
    raise RuntimeError(
        'A quantidade de ID_GLEBA2 distintos não corresponde ao esperado.'
    )

print('VALIDAÇÃO APROVADA')
print('Registros preservados:', f'{total_validado:,}')
print('ID_GLEBA2 distintos:', f'{len(ids_gleba2_distintos):,}')
print('Ocorrências adicionais que reutilizaram um ID:', f'{total_ocorrencias_repetidas:,}')
print('Divergências encontradas:', f'{divergencias:,}')

VALIDAÇÃO APROVADA
Registros preservados: 480,679
ID_GLEBA2 distintos: 395,575
Ocorrências adicionais que reutilizaram um ID: 85,104
Divergências encontradas: 0


In [7]:
# ============================================================
# 7. CONFERÊNCIA VISUAL
# ============================================================

amostra = pd.read_csv(
    ARQUIVO_SAIDA,
    sep=SEPARADOR,
    encoding='utf-8-sig',
    usecols=[COLUNA_ID_ORIGINAL, COLUNA_ID_NOVA, COLUNA_GEOMETRIA],
    dtype='string',
    nrows=10,
    low_memory=False
)

amostra['GEOMETRIA_RESUMIDA'] = (
    amostra[COLUNA_GEOMETRIA].str.slice(0, 100) + '...'
)

display(
    amostra[[
        COLUNA_ID_ORIGINAL,
        COLUNA_ID_NOVA,
        'GEOMETRIA_RESUMIDA'
    ]]
)

print('Arquivo final:')
print(ARQUIVO_SAIDA)

,ID_GLEBA,ID_GLEBA2,GEOMETRIA_RESUMIDA
0,1,1,"POLYGON ((-47.521674 -18.527653 0,-47.521697 -..."
1,2,2,"POLYGON ((-47.579356 -18.906067 0,-47.578942 -..."
2,3,3,"POLYGON ((-44.441404 -16.832015 0,-44.441419 -..."
3,4,4,"POLYGON ((-44.965694 -20.079657 0,-44.967058 -..."
4,5,5,"POLYGON ((-45.075154 -18.327891 0,-45.077628 -..."
5,6,6,"POLYGON ((-45.609635 -20.759815 0,-45.610045 -..."
6,7,7,"POLYGON ((-51.375768 -18.800172 0,-51.375429 -..."
7,8,8,"POLYGON ((-44.734281 -4.852751 0,-44.734393 -4..."
8,9,9,"POLYGON ((-45.691268 -21.352975 0,-45.69174 -2..."
9,10,10,"POLYGON ((-48.660171 -16.122723 0,-48.661014 -..."


Arquivo final:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv
